# AGENTS ET OUTILS (Агенты и инструменты)
---

## 1. Действие (Action)

- Tools (Инструменты)
- LLM Reasoning (Рассуждение LLM)
- Цикл ReAct (Reason + Act)

Цель: Превратить LLM из простого «собеседника», который только отвечает на вопросы, в агента, способного взаимодействовать с реальным миром:

выполнять вычисления;
искать информацию в интернете;
работать с файлами;
использовать API;
выполнять различные действия автоматически.

Введение

До этого момента наши модели ИИ были закрытыми. Они знали только информацию, на которой были обучены (например, до 2024 года);
данные, которые пользователь предоставил в запросе.

Агент — это LLM, которому дали набор инструментов.

Вместо того чтобы сразу отвечать, агент делает следующее:

- Размышляет над вопросом.
- Выбирает подходящий инструмент.
- Использует инструмент.
- Анализирует результат.
- При необходимости использует другой инструмент.
- Дает окончательный ответ.

Иными словами: У агента есть не только голова (LLM), но и руки (Tools), которыми он может выполнять реальные действия.

## 2. Фундаментальная концепция: цикл ReAct (Reason + Act)

Это стандартный способ мышления большинства современных агентов. Последовательность выглядит так:

- Thought (Мысль): Мне нужно узнать погоду в Париже. Для этого следует воспользоваться поисковым инструментом.

↓

- Action (Действие): Вызов функции поиска погоды.

↓

- Observation (Наблюдение): Полученный ответ: 15°C, дождь.

↓

- При необходимости цикл повторяется.

↓

- Final Answer (Финальный ответ): Сейчас в Париже 15°C и идет дождь.


## 3. Создание собственного Tool

Чтобы агент мог пользоваться инструментом, необходимо объяснить ему:

- что делает инструмент;
- когда его использовать;
- какие параметры принимает;
- что возвращает.

Для этого используются Python Docstrings. 

Tool может быть чем угодно:

- поисковой системой;
- API;
- функцией Python;
- калькулятором;
- базой данных;
- сервисом погоды.

In [2]:
import ollama

In [3]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatOllama(model="qwen3:4b", temperature=0)

In [1]:
from langchain_core.tools import tool

@tool
def calculateur_tva(prix_ht: float) -> float:
    """
    Вычисляет цену с НДС (20%) на основе цены без НДС.
    Полезно для финансовых расчетов.
    """
    return prix_ht * 1.20

## Передача инструментов модели

Если не сообщить LLM о существовании инструментов, она никогда ими не воспользуется. Поэтому создают список:

Теперь LLM знает: У меня есть два инструмента, которыми можно пользоваться.

In [ ]:
tools = [ calculateur_tva]

llm_with_tools = llm.bind_tools(tools)

## Определение агента

В LangGraph все узлы работают через объект MessagesState. Он содержит историю диалога. Агент:

- читает историю сообщений;
- принимает решение;
- отвечает;
- если решил вызвать Tool — возвращает специальное сообщение с вызовом инструмента.

In [ ]:
def node_agent(state: MessagesState):

    response = llm_with_tools.invoke(state["messages"])

    return {"messages": [response]}

## Построение графа

Необходимо создать граф состояний. Логика простая:

Пользователь
      │
      ▼
   Агент
      │
      ▼
Нужно использовать Tool?
      │
 ┌────┴─────┐
 │          │
Да         Нет
 │          │
 ▼          ▼
Tool       Конец
 │
 ▼
Агент

Node — это вершина графа, выполняющая действие. Например: Agent-Tool

Edge — обычное соединение между узлами.

Conditional Edge - Соединение с условием.


### Инициализация графа

In [ ]:
workflow = StateGraph(MessagesState)

### Создание узлов

In [ ]:
workflow.add_node("agent", node_agent)

workflow.add_node("tools", ToolNode(tools))

### Соединения

In [ ]:
workflow.add_edge(START, "agent")  # Сначала старт
workflow.add_conditional_edges("agent", tools_condition) # Затем условие
workflow.add_edge("tools", "agent") # Возврат после использования Tool

Конец графа подключать не нужно. ```tools_condition``` сама делает проверку. Если последнее сообщение содержит Tool Call - возвращается "tools", Если Tool не нужен - END. После этого граф компилируется.

In [ ]:
agent_app = workflow.compile()

## Запуск агента

Создается системный промпт. Пользователь задает вопрос. Создаются входные данные.

In [ ]:
system_prompt = SystemMessage(content="""Tu es un assistant qui DOIT utiliser des outils.Pour chaque question, tu DOIS utiliser un outil de recherche ou de calcul. N'utilise JAMAIS tes propres connaissances pour des faits ou des prix. Si tu ne trouves pas l'info via l'outil, dis que tu ne sais pas.""")

query = "Quel est le prix actuel de l'action NVIDIA en temps réel ? Et si j'en achète 10, combien ça fait TTC avec ton calcul ?"

inputs = {
    "messages": [
        system_prompt,
        HumanMessage(content=query)
    ]
}

response = agent_app.invoke(inputs)

print(
    response["messages"][-1].content
)

## Визуализация графа

Можно экспортировать граф в Mermaid. И сохранить. Также существует LangGraph Studio — IDE для визуального просмотра и отладки графов.

In [ ]:
mermaid_code = agent_app.get_graph().draw_mermaid()
with open("graph.mmd","w") as f:
    f.write(mermaid_code)

graph_image = agent_app.get_graph().draw_mermaid_png()

# Nœuds d'évaluation (Evaluation Nodes) — Судья (Judge)

Основные понятия:

- Цикл оценки (Evaluation Loop)
- Конструктивная критика
- Аудит данных

Цель: Превратить LLM из простого "говорящего" в надежного агента, который способен проверять собственную работу перед тем, как отправить ответ пользователю.

До этого момента агент всё еще мог:

- ошибиться;
- пропустить шаг;
- неверно интерпретировать результат.

Поэтому после окончания работы появляется Судья (Evaluator). Когда агент считает, что задача завершена, его ответ отправляется Судье. Судья принимает одно из двух решений.

- Работа выполнена хорошо. Агент действовал корректно. Ответ отправляется пользователю. Цикл заканчивается.

- Работа выполнена плохо. Агент:

    - проигнорировал инструмент;
    - нарушил правила;
    - сделал неверный вывод.

Тогда Судья отправляет его обратно на доработку. То есть появляется новая петля.

Что делает Судья?

1. Проверяет доказательства (Audit). Например, если инструмент вернул: Цена = 150 €, а агент написал Цена = 170 €, Судья это обнаружит.

2. Проверяет логику. Например, агент получил: 10 + 5 = 15, а написал 16, Судья обнаружит ошибку.

3. Выносит решение. Либо ОК, либо Нужно исправить

Теперь у агента есть:

- голова (LLM);
- руки (Tools);
- контролер качества (Evaluator).

Практическая реализация: Кроме Tools необходимо добавить узел Evaluator.

![alt text](image.png)

In [ ]:
class MyGraphState(MessagesState):

    verdict: EvaluationVerdict


class EvaluationVerdict(BaseModel):

    grade: Literal["OUI","NON"]

    critique: str

Теперь State хранит: messages + verdict. Что такое EvaluationVerdict? Это структура данных. Она описывает результат проверки. Pydantic гарантирует правильный формат ответа.

In [ ]:
workflow = StateGraph(MyGraphState)

workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)
workflow.add_node("evaluator", quality_control_node)
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent")
workflow.add_conditional_edges("evaluator", route_after_eval)
app = workflow.compile()


agent- Это основной LLM. tools - Как раньше. evaluator - Это тоже LLM. У него совершенно другой prompt. Он не отвечает пользователю.Он только оценивает. Почему нужен новый GraphState? Обычный MessagesState хранит только сообщения. Но Судья должен сохранить ещё и вердикт. Поэтому создается новый класс.

Почему нельзя передавать Судье всю историю? Судье обычно передают только необходимые данные:

- последний ответ агента;
- результаты Tool;
- иногда краткую историю.

In [ ]:
evaluator_llm = model.with_structured_output(EvaluationVerdict)

# читает сообщения;
# составляет audit prompt;
# вызывает LLM;
# сохраняет verdict.
def quality_control_node(state):

    ...

    verdict = evaluator_llm.invoke(...)

    return {
        "verdict": verdict
    }

def should_continue(state):

    last_message = state["messages"][-1]

    if last_message.tool_calls:

        return "tools"

    return "evaluator"

def route_after_eval(state):
    """
    Fonction de routage conditionnel :
    Analyse le verdict de l'évaluateur contenu dans le 'state'.
    Si le verdict est 'OUI', on dirige vers la fin (END).
    Si le verdict est 'NON', on renvoie vers l'agent pour une correction.
    """

    verdict = state.get("verdict")

    if verdict.grade == "OUI":

        return END

    return "agent"

3. Выполнение упражнения
Задача

Нужно создать агента службы поддержки (SAV — Service Après-Vente), который:

- Получает пользователя и номер заказа.
- Читает информацию из базы boutique.db.
- Вычисляет, сколько дней прошло с момента заказа.
- Узнает правила возврата по категории товара (псевдо-RAG).
- Решает, можно ли вернуть товар.
- Передает решение Evaluator, который проверяет корректность ответа.

![alt text](image-1.png)

In [ ]:
from datetime import datetime
import sqlite3
from typing import Literal

from pydantic import BaseModel, Field

from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    AIMessage
)

from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOllama(model="qwen3:4b", temperature=0)

In [11]:
@tool
def query_database(user_id: int, order_id: int) -> str:
    """
    Read boutique.db and return the order information.

    Returns:
        category
        order_date
        product
    """

    conn = sqlite3.connect("boutique.db")
    cursor = conn.cursor()

    cursor.execute(
        """
        SELECT
            product,
            category,
            order_date
        FROM orders
        WHERE
            user_id = ?
            AND id = ?
        """,
        (user_id, order_id),
    )

    row = cursor.fetchone()
    conn.close()

    if row is None:
        return "Order not found"

    product, category, order_date = row

    return (
        f"Product: {product}\n"
        f"Category: {category}\n"
        f"Order date: {order_date}"
    )

In [12]:
@tool
def query_knowledge_base(category: str):
    """
    Return the return policy according
    to the product category.
    """

    kb = {
        "Electronics":
            "14 days, sealed package",

        "Furniture":
            "30 days, -15% fee if assembled",

        "Appliances":
            "30 days, 20€ flat fee",
    }

    return kb.get(category, "Unknown category")

In [13]:
@tool
def days_since_order(order_date: str):
    """
    Calculate the number of days
    between order_date and today.
    """

    order = datetime.strptime(
        order_date,
        "%Y-%m-%d"
    )

    today = datetime.today()

    delta = today - order

    return delta.days

In [14]:
tools = [
    query_database,
    query_knowledge_base,
    days_since_order,
]

tool_node = ToolNode(tools)
llm_with_tools = llm.bind_tools(tools)

In [ ]:
class EvaluationVerdict(BaseModel):

    grade: Literal["OUI", "NON"] = Field(
        description="Final verdict"
    )

    critique: str = Field(
        description="Reason"
    )

class MyGraphState(MessagesState):

    verdict: EvaluationVerdict | None

In [ ]:
system_prompt = SystemMessage(
    content="""
        You are a customer support assistant.

        You MUST:

        1. Read the database.
        2. Read the return policy.
        3. Calculate the number of days.
        4. Decide whether the product
        can be returned.

        Never invent data.

        Always use tools.
        """
        )

In [ ]:
def call_model(state: MyGraphState):

    messages = state["messages"]

    response = llm_with_tools.invoke([system_prompt] + messages)

    return {"messages": [response]}

In [ ]:
evaluator_llm = llm.with_structured_output(EvaluationVerdict)

In [ ]:
def quality_control_node(state: MyGraphState):

    messages = state["messages"]

    final_answer = messages[-1].content

    prompt = SystemMessage(
        content=f"""
You are a quality auditor.

Your job is NOT to answer.

Verify that:

- database has been used

- return policy has been used

- date calculation exists

- final conclusion follows
the evidence.

Return:

OUI

or

NON

with a critique.

Final answer:

{final_answer}
"""
    )

    verdict = evaluator_llm.invoke(
        [prompt] + messages
    )

    return {
        "verdict": verdict
    }

In [ ]:
def should_continue(state: MyGraphState):

    last = state["messages"][-1]

    if last.tool_calls:
        return "tools"

    return "evaluator"

def route_after_eval(state: MyGraphState):

    verdict = state["verdict"]

    print(verdict)

    if verdict.grade == "OUI":
        return END

    return "agent"

workflow = StateGraph(MyGraphState)

workflow.add_node(
    "agent",
    call_model
)

workflow.add_node(
    "tools",
    tool_node
)

workflow.add_node(
    "evaluator",
    quality_control_node
)

workflow.add_edge(
    START,
    "agent"
)

workflow.add_conditional_edges(
    "agent",
    should_continue
)

workflow.add_edge(
    "tools",
    "agent"
)

workflow.add_conditional_edges(
    "evaluator",
    route_after_eval
)

app = workflow.compile()

In [ ]:
question = HumanMessage(
    content="""
Can user 15 return order 203?
"""
)

response = app.invoke(
    {
        "messages": [
            question
        ]
    }
)

print()

print("=" * 50)

print(
    response["messages"][-1].content
)

print("=" * 50)